In [3]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

In [17]:
import os

os.environ["PATH"] = "/Users/srijanshukla/.local/bin:" + os.environ["PATH"]

In [2]:
import os
from agents import set_tracing_export_api_key
from agents.extensions.models.litellm_model import LitellmModel

openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    print("WARNING: OPENAI_API_KEY not found. Tracing will be disabled.")
else:
    set_tracing_export_api_key(openai_api_key)
    print("OpenAI tracing configured")

os.environ['GEMINI_API_KEY'] = os.getenv('GOOGLE_API_KEY')

gemini_model = LitellmModel(
    model="gemini/gemini-2.0-flash",  
)

print("Gemini model with LitellmModel configured")

OpenAI tracing configured
Gemini model with LitellmModel configured


### Let's start by gathering the MCP params for our trader

In [4]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [18]:

market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for our researcher

In [ ]:
researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "uv", "args": ["run", "search_server.py"]}
]

### Now create the MCPServerStdio for each

In [20]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Now let's make a Researcher Agent to do market research

And turn it into a tool - remember how this works for OpenAI Agents SDK, and the difference with handoffs?

In [21]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model=gemini_model,
        mcp_servers=mcp_servers,
    )
    return researcher

In [22]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [23]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Okay, here's a summary of the latest news on Amazon based on the search results:

*   **Job Cuts:** Amazon is reportedly targeting as many as 30,000 corporate job cuts, potentially affecting HR, devices, services, and operations.
*   **International Expansion:** Amazon is making strides in its international expansion by launching Amazon Bazaar in 14 new markets.
*   **Disaster Relief:** Amazon has deployed its disaster relief technology kits internationally for the first time in response to Hurricane Melissa.


[non-fatal] Tracing client error 400: {
  "error": {
    "message": "Invalid type for 'data[0].span_data.result': expected an array of strings, but got null instead.",
    "type": "invalid_request_error",
    "param": "data[0].span_data.result",
    "code": "invalid_type"
  }
}


### Look at the trace

https://platform.openai.com/traces

In [24]:
john_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("john").reset(john_initial_strategy)

display(Markdown(await read_accounts_resource("john")))
display(Markdown(await read_strategy_resource("john")))

{"name": "john", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-11-16 01:31:39", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create our Trader Agent

In [25]:
agent_name = "john"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [26]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is john and your account is under your name, john.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader that aggressively buys and sells shares based on news and market conditions.
Your current holdings and balance is:
{"name": "john", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-11-16 01:31:39", 10000.0], ["2025-11-16 01:31:48", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.


### And to run our Trader

In [27]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model=gemini_model,
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

I have bought 100 shares of UVXY at a price of $10.83. This decision was based on my analysis of the market, which indicates increased volatility. UVXY is a volatility ETF that is expected to increase in value as market volatility rises. My balance is now $8914.83.


### trace

http://platform.openai.com/traces


In [28]:
await read_accounts_resource(agent_name)

'{"name": "john", "balance": 8914.833999999999, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"UVXY": 100}, "transactions": [{"symbol": "UVXY", "quantity": 100, "price": 10.85166, "timestamp": "2025-11-16 01:32:35", "rationale": "Market is experiencing volatility and UVXY is a volatility ETF that should increase in value as VIX increases."}], "portfolio_value_time_series": [["2025-11-16 01:31:39", 10000.0], ["2025-11-16 01:31:48", 10000.0], ["2025-11-16 01:32:35", 9997.833999999999], ["2025-11-16 01:32:50", 9997.833999999999]], "total_portfolio_value": 9997.833999999999, "total_profit_loss": -2.166000000001077}'

In [29]:
from traders import Trader

In [30]:
trader = Trader("John")

In [32]:
await trader.run()

OpenAI tracing configured
OpenAI tracing configured


In [34]:
await read_accounts_resource("John")

'{"name": "john", "balance": 6844.701999999999, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"UVXY": 100, "GME": 100}, "transactions": [{"symbol": "UVXY", "quantity": 100, "price": 10.85166, "timestamp": "2025-11-16 01:32:35", "rationale": "Market is experiencing volatility and UVXY is a volatility ETF that should increase in value as VIX increases."}, {"symbol": "GME", "quantity": 100, "price": 20.70132, "timestamp": "2025-11-16 03:34:24", "rationale": "GME is adding trading card inventory, which might be a catalyst for a short-term price increase."}], "portfolio_value_time_series": [["2025-11-16 01:31:39", 10000.0], ["2025-11-16 01:31:48", 10000.0], ["2025-11-16 01:32:35", 9997.833999999999], ["2025-11-16 01:32:50", 9997.833999999999], ["2025-11-16 03:29:51", 9997.833999999999], ["2025-11-16 03:33:52", 9997.833999999999], ["2025-11-16 03:34:24", 9993.702], ["2025-11-16 03:36:33", 9993.702]], "total_portf

### trace

https://platform.openai.com/traces


In [35]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")

We have 6 MCP servers, and 15 tools
